In [1]:
!pip install numpy pandas scikit-learn tensorflow joblib

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

df = pd.read_csv('dataset/diabetes_binary_health_indicators_BRFSS2015.csv')

# Menampilkan 5 data teratas untuk memastikan data terbaca
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
# Masukkan nama kolom yang sudah disesuaikan dengan dataset aslimu ('NoDocbcCost')
features = [
    'HighBP', 'HeartDiseaseorAttack', 'Smoker', 'HvyAlcoholConsump', 
    'PhysActivity', 'Fruits', 'Veggies', 'NoDocbcCost', 'DiffWalk', 
    'MentHlth', 'PhysHlth', 'Age', 'Sex', 'BMI'
]

# Ambil fitur dan target
X = df[features]
y = df['Diabetes_binary']

# Membagi data menjadi Data Training (80%) dan Data Testing (20%)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Fitur yang digunakan:", list(X.columns))
print("Jumlah fitur wajib:", X.shape[1])
print("Data Training:", X_train.shape)
print("Data Testing:", X_test.shape)

Fitur yang digunakan: ['HighBP', 'HeartDiseaseorAttack', 'Smoker', 'HvyAlcoholConsump', 'PhysActivity', 'Fruits', 'Veggies', 'NoDocbcCost', 'DiffWalk', 'MentHlth', 'PhysHlth', 'Age', 'Sex', 'BMI']
Jumlah fitur wajib: 14
Data Training: (202944, 14)
Data Testing: (50736, 14)


In [4]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
from xgboost import XGBClassifier
import joblib

print("Memulai pelatihan XGBoost...")

# Inisialisasi XGBoost dengan parameter dasar
xgb_model = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

print("\nSedang melatih model XGBoost, mohon tunggu...")
xgb_model.fit(X_train_scaled, y_train)
print("Model berhasil dilatih!")

Memulai pelatihan XGBoost...

Sedang melatih model XGBoost, mohon tunggu...


c:\Users\Ezra\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:09:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model berhasil dilatih!


In [8]:
y_pred = xgb_model.predict(X_test_scaled)

akurasi = accuracy_score(y_test, y_pred)
print(f"\nAkurasi Model pada Data Uji: {round(akurasi * 100, 2)}%")
print("\nLaporan Klasifikasi Lengkap:")
print(classification_report(y_test, y_pred))


Akurasi Model pada Data Uji: 86.25%

Laporan Klasifikasi Lengkap:
              precision    recall  f1-score   support

         0.0       0.87      0.98      0.92     43667
         1.0       0.53      0.11      0.18      7069

    accuracy                           0.86     50736
   macro avg       0.70      0.55      0.55     50736
weighted avg       0.82      0.86      0.82     50736



In [10]:
joblib.dump(xgb_model, "model_xgb_treehealthy.pkl")
joblib.dump(scaler, "scaler_treehealthy.pkl")

print("\nSukses! File model_xgb_treehealthy.pkl dan scaler_treehealthy.pkl telah diperbarui!")


Sukses! File model_xgb_treehealthy.pkl dan scaler_treehealthy.pkl telah diperbarui!
